# Pilot: every stage at tiny scale (paths, versions, memory, throughput)

In [ ]:
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/muttaqi123/small-model-alignment.git"
COMMIT = "5713785"
WORK = Path("/kaggle/working/small-model-alignment")
os.chdir("/kaggle/working")
if not WORK.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(WORK)], check=True)
subprocess.run(["git", "-C", str(WORK), "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", str(WORK), "checkout", "-q", COMMIT], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = str(WORK)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


def run(cmd, env=None):
    """Run a shell command, stream its output, and fail the notebook on error."""
    print(f"$ {cmd}", flush=True)
    merged = {**os.environ, **(env or {})}
    proc = subprocess.Popen(cmd, shell=True, env=merged, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {cmd}")


def start(cmd, log_path, env=None):
    """Start a background command (e.g. generation on the second GPU)."""
    print(f"$ {cmd} > {log_path} &", flush=True)
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    handle = open(log_path, "w")
    merged = {**os.environ, **(env or {})}
    return subprocess.Popen(cmd, shell=True, env=merged, stdout=handle, stderr=subprocess.STDOUT, text=True), log_path


def wait(job):
    proc, log_path = job
    code = proc.wait()
    print(Path(log_path).read_text()[-3000:])
    if code != 0:
        raise RuntimeError(f"background job failed: {log_path}")


def archive(zip_path, items, exclude=("trainer", "best", "final", "adapter")):
    """Zip small artifacts (metrics, configs, generations, plots); never weights."""
    import zipfile
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for item in items:
            p = Path(item)
            files = [p] if p.is_file() else sorted(f for f in p.rglob("*") if f.is_file())
            for f in files:
                if any(part in exclude for part in f.parts) or f.suffix in (".safetensors", ".bin", ".pt"):
                    continue
                zf.write(f, str(f))
    print(f"{zip_path}: {Path(zip_path).stat().st_size / 1e6:.1f} MB")


def find_input(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"no input matches {pattern}")
    print(f"found {pattern} -> {hits[0]}")
    return hits[0]

In [ ]:
run("nvidia-smi")
run("python -m pip uninstall -y -q torchao || true")
run('python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib "trl>=0.19,<0.24" "peft>=0.15,<0.18"')
run("python -m pip install -q -e . --no-deps")
run('python -c "import torch, transformers, datasets; print(\'torch\', torch.__version__, \'| transformers\', transformers.__version__, \'| datasets\', datasets.__version__, \'| cuda\', torch.version.cuda, \'| gpus\', torch.cuda.device_count())"')
run("python -m pytest -q tests/")

In [ ]:
run("python scripts/prepare_data.py --out_dir outputs/data")
run("python -m alignment.sft --train_path outputs/data/sft_train.jsonl --val_path outputs/data/sft_val.jsonl "
    "--output_dir outputs/pilot/sft --max_train_examples 320 --max_val_examples 64 --eval_every 5")

In [ ]:
import json
for line in open("outputs/pilot/sft/metrics.jsonl"):
    print(line.strip())

In [ ]:
run("python -m alignment.dpo --sft_path outputs/pilot/sft/best --train_path outputs/data/dpo_hh_train.jsonl "
    "--val_path outputs/data/dpo_hh_val.jsonl --output_dir outputs/pilot/dpo_hh --max_train_examples 320 "
    "--max_val_examples 64 --eval_every 5")
run("python -m alignment.dpo --sft_path outputs/pilot/sft/best --train_path outputs/data/dpo_uf_train.jsonl "
    "--val_path outputs/data/dpo_uf_val.jsonl --output_dir outputs/pilot/dpo_uf --max_train_examples 96 "
    "--max_val_examples 32 --eval_every 1")

In [ ]:
run(f"python -m alignment.generate --batch_size 16 --max_new_tokens 512 --repetition_penalty 1.1 --model_path Qwen/Qwen2.5-0.5B --eval_path eval_sets/alpaca_eval_300_seed42.json --limit 4 "
    "--output_path outputs/pilot/gen_base.json --generator qwen2.5-0.5b-base --dataset alpaca_eval")
run(f"python -m alignment.generate --batch_size 16 --max_new_tokens 512 --repetition_penalty 1.1 --model_path outputs/pilot/sft/best --eval_path eval_sets/hh_rlhf_eval_300_seed42.json --limit 4 "
    "--output_path outputs/pilot/gen_sft.json --generator pilot-sft --device cuda:1")
for f in ("outputs/pilot/gen_base.json", "outputs/pilot/gen_sft.json"):
    for r in json.load(open(f)):
        print("=" * 80, "\n", r["instruction"][:300], "\n---\n", r["output"][:1500])

In [ ]:
run("python scripts/train_lora_dpo.py --output_dir outputs/pilot/lora --configs r16_all_lr5e-5 --epochs 1",
    env={"CUDA_VISIBLE_DEVICES": "0"})
run(f"python -m alignment.generate --batch_size 16 --max_new_tokens 512 --repetition_penalty 1.1 --model_path HuggingFaceTB/SmolLM2-360M-Instruct --adapter_path outputs/pilot/lora/r16_all_lr5e-5/adapter "
    "--prompt_format chat_template --eval_path eval_sets/persona_style_eval_40.json --limit 4 "
    "--output_path outputs/pilot/gen_lora.json --generator pilot-lora")
for r in json.load(open("outputs/pilot/gen_lora.json")):
    print("=" * 80, "\n", r["instruction"], "\n---\n", r["output"][:800])

In [ ]:
run("python scripts/plot_metrics.py --sft outputs/pilot/sft --dpo outputs/pilot/dpo_hh outputs/pilot/dpo_uf --lora outputs/pilot/lora --out_dir outputs/pilot/plots")
shutil.rmtree("outputs/pilot/sft/final", ignore_errors=True)
for d in glob.glob("outputs/pilot/dpo_*/*/"):
    shutil.rmtree(d, ignore_errors=True)
shutil.rmtree("outputs/data", ignore_errors=True)
run("du -sh outputs/pilot/*")